# GLiNER fine-tune — Bosch GDPR PII (Colab, zero-touch)

## What this does
Fine-tunes `urchade/gliner_multi_pii-v1` (200M params) on your Bosch synthetic PII data.
~30-60min on free T4 GPU. Trained model saved to your Google Drive.

## Before running

1. **GPU**: Runtime → Change runtime type → **T4 GPU** → Save
2. **Prep zips on your Mac**:
   ```bash
   cd /Users/mohannedkandil/conductor/workspaces/tracer/west-monroe
   git archive --format=zip HEAD -o /tmp/tracer.zip
   zip -r /tmp/gdpr-data.zip data/out data/gliner
   ```
3. Run cells 1 → 7 in order. Cell 1 will pop a **file picker** to upload both zips.

In [ ]:
# 1. GPU check (skip if you already see GPU info)
!nvidia-smi 2>&1 | head -10
import os
os.chdir('/content')
print('cwd:', os.getcwd())

In [ ]:
# 2. Upload zips (interactive file picker — choose tracer.zip AND gdpr-data.zip)
import os, shutil
os.chdir('/content')
from google.colab import files

REPO_DIR = '/content/repo'

# If repo already extracted, skip the upload step
if os.path.isfile(f'{REPO_DIR}/train/finetune.py') and os.path.isfile(f'{REPO_DIR}/data/gliner/train.jsonl'):
    print('Already extracted — skipping upload.')
else:
    print('Pick both zip files: tracer.zip AND gdpr-data.zip (multi-select with Shift)')
    uploaded = files.upload()  # blocks until user picks files
    for fname in uploaded:
        print(f'Got {fname}: {len(uploaded[fname])} bytes')

    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    os.makedirs(REPO_DIR, exist_ok=True)

    for fname in uploaded:
        if fname.endswith('.zip'):
            print(f'Extracting {fname} → {REPO_DIR}')
            !unzip -q -o "/content/$fname" -d $REPO_DIR

# Verify
import sys
for p in ['train/__init__.py', 'train/finetune.py', 'data_gen/__init__.py', 'pyproject.toml', 'data/gliner/train.jsonl', 'data/gliner/val.jsonl']:
    full = os.path.join(REPO_DIR, p)
    assert os.path.isfile(full), f'MISSING after extract: {full}'
print('✅ Repo + data ready at', REPO_DIR)

# Make importable from any cell
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR
os.chdir(REPO_DIR)

In [ ]:
# 3. Install training deps (torch already on Colab)
import os, sys
REPO_DIR = '/content/repo'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

!pip install -q 'gliner>=0.2.13' 'transformers>=4.40.0' 'accelerate>=0.30.0' 'seqeval>=1.2.2' orjson faker jinja2 pydantic httpx python-dotenv tqdm

import gliner, transformers, accelerate, torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'gliner={gliner.__version__}  transformers={transformers.__version__}  accelerate={accelerate.__version__}')

In [ ]:
# 4. Verify GLiNER-format data (uploaded zip should already include data/gliner/)
import os
REPO_DIR = '/content/repo'
os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

needed = ['data/gliner/train.jsonl', 'data/gliner/val.jsonl']
missing = [p for p in needed if not os.path.isfile(os.path.join(REPO_DIR, p))]
if missing:
    print(f'Missing: {missing} — running convert from data/out/')
    assert os.path.isfile(f'{REPO_DIR}/data/out/train.jsonl'), 'data/out/train.jsonl also missing — re-zip data/out + re-upload'
    !PYTHONPATH=$REPO_DIR python -m train.convert --in-dir data/out --out-dir data/gliner
else:
    print('✅ data/gliner/ ready')
!ls -lh $REPO_DIR/data/gliner/

In [ ]:
# 5. Mount Google Drive (checkpoint survives session disconnect)
import os
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/gliner-bosch'
os.makedirs(OUT_DIR, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

In [ ]:
# 6. Train (~30-60min on T4)
import os
REPO_DIR = '/content/repo'
os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

assert os.path.isfile(f'{REPO_DIR}/train/finetune.py'), 'cell 2 not run'
assert os.path.isfile(f'{REPO_DIR}/data/gliner/train.jsonl'), 'cell 4 not run'
assert 'OUT_DIR' in dir(), 'cell 5 not run (Drive mount)'

print(f'cwd={os.getcwd()}')
print(f'OUT_DIR={OUT_DIR}')

!PYTHONPATH=$REPO_DIR python -m train.finetune \
    --train data/gliner/train.jsonl \
    --val data/gliner/val.jsonl \
    --base-model urchade/gliner_multi_pii-v1 \
    --out $OUT_DIR \
    --epochs 3 \
    --batch-size 16 \
    --lr 5e-6 \
    --lr-others 1e-5

In [ ]:
# 7. Sanity-check the trained model on a held-out example
from gliner import GLiNER
model = GLiNER.from_pretrained(OUT_DIR, local_files_only=True)
labels = ['PERSON', 'EMPLOYEE_ID', 'EMAIL', 'PHONE', 'ADDRESS', 'TAX_ID', 'IBAN', 'DEPARTMENT', 'COMPANY', 'DATE', 'SIGNATURE', 'USERNAME']
sample = (
    'Spesenabrechnung (ausgefüllt)\n'
    'Mitarbeiter: Hans Müller (E-43217)\n'
    'Abteilung: Forschung & Entwicklung\n'
    'Datum: 12 Mär 2026\n'
    'Vorgesetzter: Anna Becker\n'
    'Unterschrift: A. Becker'
)
for ent in model.predict_entities(sample, labels, threshold=0.4):
    print(f"{ent['label']:<14} {ent['text']!r}  (score={ent['score']:.2f})")

In [ ]:
# 8. Zip + download checkpoint for use in scan service
import shutil
shutil.make_archive('/content/gliner-bosch', 'zip', OUT_DIR)
from google.colab import files
files.download('/content/gliner-bosch.zip')